In [120]:
# Quarterly depedency graph

In [121]:
import os
os.getcwd()

'C:\\Users\\ugne.keliauskaite\\Bruegel\\Research - 2021-11 European natural gas imports\\Data'

In [122]:
#Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data' # Gio
Share_point = r'C:\Users\ugne.keliauskaite\Bruegel\Research - 2021-11 European natural gas imports\Data' # Ugne
os.chdir(Share_point)

In [123]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

In [124]:
# import ENTSOG pipeline data
entsog = pd.read_csv(r'Imports\EU27\df1.csv')
del entsog['dates.1']
entsog = entsog.set_index(pd.DatetimeIndex(entsog['dates']))
del entsog['dates']

In [125]:
# import LNG data from GIE (we only know where the LNG arrives, not where it comes from)
# agsi = pd.read_csv(r'C:\\Users\\giovanni.sgaravatti\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Gio
agsi = pd.read_csv(r'C:\\Users\\ugne.keliauskaite\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Ugne
agsi=agsi.set_index(pd.DatetimeIndex(agsi['dates']))
del agsi['dates']
del agsi['index']

In [126]:
# import Bloomberg LNG data (with these data we know both where it comes from and where it arrives, but we trust GIE better - also to be consistent with the tracker)
lng_b = pd.read_excel(r'LNG\Bloomberg\granular LNG imports.xlsx') # Gio
lng_b.rename(columns= {'Unnamed: 0':'dates'},inplace=True)
lng_b = lng_b.set_index(pd.DatetimeIndex(lng_b['dates']))
del lng_b['dates']

In [127]:
lng_b['Tot'] = lng_b.sum(axis=1,numeric_only=True)

In [128]:
# Divide each column by the 'Tot' column
ratios_df = lng_b.div(lng_b['Tot'], axis=0)

In [129]:
agsi_m = agsi.groupby(pd.Grouper(freq='M'))['sendOut'].sum(numeric_only=True)
# take only values after 2019 to be consistent with Bloomberg data
agsi_19 = agsi_m['2019':]

In [130]:
# to align with Bloombgerg lng
agsi_19= agsi_19.iloc[:-1] #Ugne change this one!

In [131]:
agsi_19

dates
2019-01-31     65357.1
2019-02-28     59014.4
2019-03-31     84049.6
2019-04-30     88014.5
2019-05-31     80457.8
                ...   
2023-11-30    121406.5
2023-12-31    117767.6
2024-01-31    108864.5
2024-02-29     98475.6
2024-03-31    108550.2
Freq: M, Name: sendOut, Length: 63, dtype: float64

In [132]:
# create new dataframe
lng = pd.DataFrame()
lng['dates'] = agsi_19.index

In [133]:
agsi_19

dates
2019-01-31     65357.1
2019-02-28     59014.4
2019-03-31     84049.6
2019-04-30     88014.5
2019-05-31     80457.8
                ...   
2023-11-30    121406.5
2023-12-31    117767.6
2024-01-31    108864.5
2024-02-29     98475.6
2024-03-31    108550.2
Freq: M, Name: sendOut, Length: 63, dtype: float64

In [134]:
 # multiply Bloomberg LNG ratios by AGSI totals
# and convert to M3m
for column in ratios_df.columns:
    lng[column] = agsi_19.values*ratios_df[column].values/10.3

In [135]:
lng.set_index(pd.DatetimeIndex(lng['dates']),inplace=True)
del lng['dates']

In [136]:
lng['Total less Russia and USA'] = lng['Tot'] - lng['Russia'] - lng['United States']

In [137]:
# Change the months for which you have data here
months = pd.date_range(start='2019-01-01', end='2024-03-31', freq='M')

In [138]:
entsog = entsog['2019':]

In [139]:
converter = 10300000   ## KWh to M3m --> 10.3 KWh/m^3     # on ENTSOG/AGSI the data comes in KWh, we transform it (later on) in M3m 

In [140]:
entsog_m = pd.DataFrame()
entsog_m['dates'] = months
entsog_m.set_index(pd.DatetimeIndex(entsog_m['dates']),inplace=True)
del entsog_m['dates']

for country in ['Russia', 'Norway','Algeria', 'UK', 'Azerbaijan','Libya']:
    entsog_m[country] = entsog[entsog['aggregation'] == country]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [141]:
for pipe in ['Ukraine Gas Transit', 'Yamal (BY,PL)','Nord Stream', 'Turkstream']:
    entsog_m[pipe] = entsog[entsog['aggregation2'] == pipe]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [142]:
entsog_q = entsog_m.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)
lng_q = lng.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)

In [143]:
graph = entsog_q
graph['USA LNG'] = lng_q['United States']
graph['Russia LNG'] = lng_q['Russia']
del graph['Russia']
graph['LNG less RU and USA'] = lng_q['Total less Russia and USA']
graph = graph['2021':]

In [144]:
graph.tail()

,Norway,Algeria,UK,Azerbaijan,Libya,Ukraine Gas Transit,"Yamal (BY,PL)",Nord Stream,Turkstream,USA LNG,Russia LNG,LNG less RU and USA
dates,,,,,,,,,,,,
2023-03-31,23518.528435,7318.676874,4850.594403,3068.134926,691.298139,2837.380096,0.0,0.0,2637.472494,14101.048245,5036.417725,13170.213642
2023-06-30,22424.073815,8509.097454,6484.323957,3037.083973,722.073983,3267.948682,0.0,0.0,2588.875575,16794.255840,4601.492368,14550.795481
2023-09-30,20593.586580,8922.806611,3562.232558,3050.647476,506.069451,3248.204904,0.0,0.0,4388.088996,14462.264634,3878.911022,12677.183567
2023-12-31,23863.828909,8210.283462,3141.930247,3233.510731,670.741001,4261.379134,0.0,0.0,4071.679780,17030.290121,4171.789277,12172.804097
2024-03-31,24152.453176,7446.441824,1869.480192,3202.055103,481.921461,3976.025982,0.0,0.0,3903.919757,15637.682202,5838.909731,9192.369232


In [145]:
graph = graph[['Nord Stream','Yamal (BY,PL)','Ukraine Gas Transit','Turkstream','Russia LNG', 'USA LNG','LNG less RU and USA', 'Norway','Algeria','UK','Azerbaijan','Libya']]

In [146]:
from datetime import datetime
today = date.today()

In [147]:
with pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today)) as writer:
    Excelwriter = pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today),engine="xlsxwriter")
    entsog_q.to_excel(Excelwriter, sheet_name="ENTSOG", index=True)
    lng_q.to_excel(Excelwriter, sheet_name="LNG", index=True)
    graph.to_excel(Excelwriter, sheet_name="graph", index=True)
Excelwriter.close()
Excelwriter.save()

C:\Users\ugne.keliauskaite\AppData\Local\Temp\ipykernel_30572\2207928906.py:7: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  Excelwriter.save()
c:\Users\ugne.keliauskaite\AppData\Local\anaconda3\Lib\site-packages\xlsxwriter\workbook.py:368: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
